In [1]:
import torch
import json
import os
import logging
from datetime import datetime
from typing import List, Dict, Any, Optional
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import warnings
import re
warnings.filterwarnings('ignore')

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [2]:
# Cell 2: Device setup and configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔧 Using device: {device}")

# Main configuration
CONFIG = {
    "model_id": "meta-llama/Meta-Llama-3-8B-Instruct",  # Use Instruct version
    "use_quantization": True,
    "device": device,
    "max_new_tokens": 300,
    "temperature": 0.3,
    "input_file": "output_entities2.json",
    "output_dir": "lulc_extraction_output",
    "max_sentences": 20,  # Process first 50 sentences for testing
}

# Valid LULC relations
VALID_RELATIONS = [
    "CHANGE_TO", "INCREASES_BY", "DECREASES_BY", "CAUSES", "LOCATED_IN",
    "OCCURS_DURING", "MEASURES", "AFFECTS", "FROM_TO", "ENABLES"
]

# Create output directory
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(f" Configuration loaded successfully")
print(f" Output directory: {CONFIG['output_dir']}")
print(f" Model: {CONFIG['model_id']}")

🔧 Using device: cuda
 Configuration loaded successfully
 Output directory: lulc_extraction_output
 Model: meta-llama/Meta-Llama-3-8B-Instruct


In [3]:
# Cell 3: Data loading function
def load_preprocessed_data(file_path: str) -> List[Dict]:
    """Load already processed data with sentence and entities keys"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print(f" Loaded {len(data)} items from {file_path}")
        
        processed_data = []
        for item in data:
            # Clean the sentence text
            sentence = item.get('sentence', '')
            if sentence.startswith("text': '"):
                sentence = sentence[7:]  # Remove "text': '"
            if sentence.endswith("'"):
                sentence = sentence[:-1]  # Remove trailing quote
            
            # Get existing entities (if any)
            entities = item.get('entities', [])
            
            processed_data.append({
                'sentence': sentence,
                'entities': entities,
                'original_data': item
            })
        
        # Print statistics
        total_entities = sum(len(item['entities']) for item in processed_data)
        sentences_with_entities = sum(1 for item in processed_data if item['entities'])
        
        print(f"📊 Processing Statistics:")
        print(f"  - Total sentences: {len(processed_data)}")
        print(f"  - Sentences with entities: {sentences_with_entities}")
        print(f"  - Total entities extracted: {total_entities}")
        if len(processed_data) > 0:
            print(f"  - Average entities per sentence: {total_entities/len(processed_data):.2f}")
        
        return processed_data
        
    except Exception as e:
        logger.error(f"Error loading data: {e}")
        return []

In [4]:
# Cell 4: Model loading function
def load_llama_model(model_id: str, use_quantization: bool = True):
    """Load Llama model with proper configuration"""
    print(f" Loading model: {model_id}")
    
    try:
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        print(f"📝 Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")
        
        # Configure quantization for memory efficiency
        quantization_config = None
        if use_quantization and device.type == "cuda":
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True
            )
            print("🔧 Using 4-bit quantization")
        
        # Load model
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto" if device.type == "cuda" else None,
            quantization_config=quantization_config,
            torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
            trust_remote_code=True
        )
        
        print(f"✅ Model loaded successfully on {model.device}")
        return model, tokenizer
        
    except Exception as e:
        logger.error(f"Failed to load model: {e}")
        raise

print(" Model loading function defined")

 Model loading function defined


In [11]:
def build_entity_recognition_prompt(sentence):
    """Build prompt for LULC entity recognition only (Task 1)"""
    
    system_message = """You are an expert in Land Use Land Cover (LULC) analysis. Your ONLY task is to identify and classify entities from the given sentence. DO NOT extract relationships.

ENTITY TYPES (USE ONLY THESE):

🌍 **LOC**: Geographical locations at any scale
- Countries: Brazil, Senegal, Rwanda
- Regions: Sahel, West Africa, Central Africa  
- Cities/Places: Niamey, Shanghai, Saloum
- Study sites: study site, regional area

🌱 **LULC**: Land cover types and land use categories
- Natural: forest, woodland, grassland, savanna, tree savanna
- Agricultural: cropland, cropped fields, farmland, cultivated land, fallows
- Urban: urban area, built-up area, settlements
- Degraded: degraded land, degraded savanna, bare soil
- Vegetation: woody vegetation, herbaceous vegetation

📈 **CHANGE**: Words indicating measurable differences or transformations
- Direction: increase, decrease, loss, gain, growth
- Action: converted, expanded, reduced, degraded
- Process: transformation, expansion, decline

⚡ **PROCESS**: Natural phenomena and human activities
- Natural: drought, desertification, fire
- Human: cultivation, grazing, irrigation, deforestation
- Agricultural: weeding, harvesting, agroforestry

📅 **DATE**: Specific years or time points
- Examples: 1990, 2020, 1970s, 1980s
- NOT time periods: use TIME_PERIODS for ranges

⏰ **TIME_PERIODS**: Time spans with beginning and end
- Examples: 1990-2000, between 1995 and 2005, from 2010 to 2020
- past three decades, 16 years

📊 **PERCENT**: Percentage values or equivalent expressions
- Examples: 30%, 51.3%, half, quarter
- Must include % symbol or equivalent word

🔢 **QUANTITY**: Numerical values without specific units
- Examples: 0.8, 1.6, 2, 5 (ratios)
- Pure numbers: 26, 31, 10

📏 **SURFACE_UNIT**: Area measurements with units
- Examples: 3 million km², 30 km², 45 km², hectares, acres

🗺️ **COORDINATES**: Geographic coordinate expressions
- Examples: 40.7°N, latitude 23.5, 80 km (distance)

EXTRACTION RULES:
1. ✂️ **Be concise**: Extract minimal meaningful text
   - ✅ "loss" not "observed loss of woody vegetation"
   - ✅ "forest" not "forest land area"

2. 📝 **List dates separately**: 
   - ✅ "1970s" and "1980s" as two separate entities
   - ✅ "1996" and "2017" as separate entities

3. 🎯 **Key distinctions**:
   - "desertification" = PROCESS (the process of becoming desert)
   - "woody vegetation" = LULC (the vegetation itself)
   - "converted" = CHANGE (the transformation action)

4. 🚫 **Avoid over-extraction**:
   - Don't extract common words like "area", "land", "cover" unless they're part of a specific LULC type
   - Focus on meaningful content words

EXAMPLES:

**Sentence**: "Between 1995 and 2005, 30% of Brazil's forest area was converted to cropland due to agricultural expansion."

**Entities**:
- Between 1995 and 2005 | TIME_PERIODS
- 30% | PERCENT  
- Brazil | LOC
- forest | LULC
- converted | CHANGE
- cropland | LULC
- agricultural expansion | PROCESS

**Sentence**: "Urban area expanded from 30 km² in 1990 to 45 km² in 2020 in Shanghai."

**Entities**:
- Urban area | LULC
- expanded | CHANGE
- 30 km² | SURFACE_UNIT
- 1990 | DATE
- 45 km² | SURFACE_UNIT  
- 2020 | DATE
- Shanghai | LOC

**OUTPUT FORMAT**:
For each entity found, use this exact format:
- entity_text | ENTITY_TYPE

IMPORTANT: 
- Extract ONLY entities, NO relationships
- Use ONLY the entity types listed above
- Be systematic: scan the sentence for each entity type
- Include ALL relevant entities, don't miss any"""

    # Llama 3 format with special tokens
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{system_message}<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Extract ONLY the entities from this sentence: "{sentence}"

Think step-by-step:
1. Scan for LOC (geographical locations)
2. Scan for LULC (land cover/use types)  
3. Scan for CHANGE (transformation words)
4. Scan for PROCESS (natural/human activities)
5. Scan for temporal entities (DATE, TIME_PERIODS)
6. Scan for quantitative entities (PERCENT, QUANTITY, SURFACE_UNIT, COORDINATES)

ENTITIES:<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
ENTITIES:
-"""
    
    return prompt

# Test function
def test_entity_recognition():
    """Test the entity recognition prompt with sample sentences"""
    
    test_sentences = [
        "After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often considered as irreversible desertification and large parts of the Sahel were designated as degraded land",
        "From 1996 to 2017, the area covered by cropped fields has increased from 40% to 51.3% over the study site area",
        "Urban area expanded from 30 km² in 1990 to 45 km² in 2020 in Shanghai"
    ]
    
    for i, sentence in enumerate(test_sentences):
        print(f"\n=== TEST SENTENCE {i+1} ===")
        print(f"Sentence: {sentence}")
        print(f"\nPrompt generated:")
        print(build_entity_recognition_prompt(sentence))
        print("-" * 80)

test_entity_recognition()


=== TEST SENTENCE 1 ===
Sentence: After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often considered as irreversible desertification and large parts of the Sahel were designated as degraded land

Prompt generated:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert in Land Use Land Cover (LULC) analysis. Your ONLY task is to identify and classify entities from the given sentence. DO NOT extract relationships.

ENTITY TYPES (USE ONLY THESE):

🌍 **LOC**: Geographical locations at any scale
- Countries: Brazil, Senegal, Rwanda
- Regions: Sahel, West Africa, Central Africa  
- Cities/Places: Niamey, Shanghai, Saloum
- Study sites: study site, regional area

🌱 **LULC**: Land cover types and land use categories
- Natural: forest, woodland, grassland, savanna, tree savanna
- Agricultural: cropland, cropped fields, farmland, cultivated land, fallows
- Urban: urban area, built-up area, settlements
- Degraded: degraded land, deg

In [6]:
def generate_lulc_extraction(sentence, model, tokenizer):
    """Generate entity and relation extraction using Llama 3"""
    prompt = build_lulc_extraction_prompt(sentence)
    
    # Tokenize with proper settings for Llama 3
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,  # Enable truncation for safety
        max_length=4096,    # Llama 3 8B context window
        padding=True,
        return_attention_mask=True
    )
    
    # Move to device
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Generate with Llama 3 optimized settings
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            temperature=CONFIG["temperature"],
            do_sample=True,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )
    
    # Decode and return
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    
    return response

print(" Generation function defined")

 Generation function defined


In [8]:
def clean_and_parse_response(response: str, original_sentence: str) -> Dict[str, Any]:
    """Improved parsing that handles both colon and pipe separators"""
    
    result = {
        'entities': [],
        'relations': [],
        'raw_response': response
    }
    
    if not response:
        return result
    
    lines = response.split('\n')
    current_section = "ENTITIES"
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
            
        # Check for section transitions
        if line.upper().startswith("RELATIONS"):
            current_section = "RELATIONS"
            continue
            
        # Parse entities - handles both formats
        if current_section == "ENTITIES":
            # Remove leading dashes/bullets
            clean_line = line.lstrip('-•· ').strip()
            
            # Try colon format: "text:TYPE"
            if ':' in clean_line and not '|' in clean_line:
                parts = clean_line.split(':', 1)
                text = parts[0].strip()
                entity_type = parts[1].strip()
                result['entities'].append({'text': text, 'type': entity_type})
                
            # Try pipe format: "text | TYPE"
            elif '|' in clean_line:
                parts = clean_line.split('|', 1)
                text = parts[0].strip()
                entity_type = parts[1].strip()
                result['entities'].append({'text': text, 'type': entity_type})
                
            # Fallback: Check if line contains a valid entity type
            else:
                valid_types = ['CHANGE', 'LOC', 'LULC', 'DATE', 'PERCENT', 'CARDINAL', 
                              'COORDINATES', 'SURFACE_UNIT', 'PROCESS', 'QUANTITY']
                for etype in valid_types:
                    if etype in line:
                        text = line.replace(etype, '').strip(' :-|')
                        if text:
                            result['entities'].append({'text': text, 'type': etype})
                            
        # Parse relations
        elif current_section == "RELATIONS":
            if "--" in line:
                relation_text = line.lstrip('-•· ').strip()
                result['relations'].append(relation_text)
    
    # Extract missing PERCENT entities
    percentage_pattern = r'\b\d+\.?\d*%\b'
    for relation in result['relations']:
        percentages = re.findall(percentage_pattern, relation)
        for pct in percentages:
            pct_exists = any(entity['text'] == pct for entity in result['entities'])
            if not pct_exists and pct in original_sentence:
                result['entities'].append({'text': pct, 'type': 'PERCENT'})
    
    # Remove duplicates
    seen = set()
    unique_entities = []
    for entity in result['entities']:
        ident = (entity['text'].lower(), entity['type'])
        if ident not in seen:
            seen.add(ident)
            unique_entities.append(entity)
            
    result['entities'] = unique_entities
    return result

# Test with your model output
test_response = """ After the droughts in the 1970s and 1980s | TIME PERIOD
- loss of woody vegetation cover | CHANGE
- woody vegetation cover | LULC
- Sahel | LOC
- degraded land | LULC

RELATIONS:
- droughts in the 1970s and 1980s:DATE --CAUSES-- loss of woody vegetation cover:CHANGE | HIGH
- loss of woody vegetation cover:CHANGE --AFFECTS-- woody vegetation cover:LULC | HIGH
- Sahel:LOC --LOCATED_IN-- loss of woody vegetation cover:CHANGE | HIGH
- loss of woody vegetation cover:CHANGE --LEADS_TO-- degraded land:LULC | HIGH"""

test_sentence = "After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often considered as irreversible and large parts of the Sahel"

parsed = clean_and_parse_response(test_response, test_sentence)
print(" Parsing Results:")
print(f"Entities: {len(parsed['entities'])}")
for e in parsed['entities']:
    print(f" - '{e['text']}' | {e['type']}")
print(f"\nRelations: {len(parsed['relations'])}")
for r in parsed['relations']:
    print(f" - {r}")

 Parsing Results:
Entities: 5
 - 'After the droughts in the 1970s and 1980s' | TIME PERIOD
 - 'loss of woody vegetation cover' | CHANGE
 - 'woody vegetation cover' | LULC
 - 'Sahel' | LOC
 - 'degraded land' | LULC

Relations: 4
 - droughts in the 1970s and 1980s:DATE --CAUSES-- loss of woody vegetation cover:CHANGE | HIGH
 - loss of woody vegetation cover:CHANGE --AFFECTS-- woody vegetation cover:LULC | HIGH
 - Sahel:LOC --LOCATED_IN-- loss of woody vegetation cover:CHANGE | HIGH
 - loss of woody vegetation cover:CHANGE --LEADS_TO-- degraded land:LULC | HIGH


In [9]:
def process_sentences_batch(sentences: List[Dict], model, tokenizer, max_sentences: int = None) -> List[Dict]:
    """Process multiple sentences and extract LULC information"""
    
    if max_sentences:
        sentences = sentences[:max_sentences]
    
    results = []
    
    print(f" Processing {len(sentences)} sentences...")
    
    for i, item in enumerate(sentences, 1):
        sentence = item['sentence']
        
        if len(sentence) < 10:  # Skip very short sentences
            continue
            
        print(f"📝 Processing {i}/{len(sentences)}: {sentence[:60]}...")
        
        try:
            # Generate extraction
            raw_response = generate_lulc_extraction(sentence, model, tokenizer)
            
            # Parse and clean
            parsed_result = clean_and_parse_response(raw_response, sentence)
            
            # Combine with original data
            result = {
                'sentence': sentence,
                'original_entities': item.get('entities', []),
                'extracted_entities': parsed_result['entities'],
                'extracted_relations': parsed_result['relations'],
                'raw_model_response': parsed_result['raw_response'],
                'processing_timestamp': datetime.now().isoformat()
            }
            
            results.append(result)
            
            # Print progress
            if parsed_result['entities']:
                print(f"    Found {len(parsed_result['entities'])} entities, {len(parsed_result['relations'])} relations")
            else:
                print(f"    No entities found")
                
        except Exception as e:
            logger.error(f"Error processing sentence {i}: {e}")
            result = {
                'sentence': sentence,
                'original_entities': item.get('entities', []),
                'extracted_entities': [],
                'extracted_relations': [],
                'error': str(e),
                'processing_timestamp': datetime.now().isoformat()
            }
            results.append(result)
            continue
            
        # Save intermediate results every 10 sentences
        if i % 10 == 0:
            save_results(results, f"{CONFIG['output_dir']}/intermediate_results_{i}.json")
    
    return results

print("✅ Batch processing function defined")

# Cell 9: Save results function
def save_results(results: List[Dict], output_path: str):
    """Save processing results to JSON file"""
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        print(f" Saved {len(results)} results to {output_path}")
    except Exception as e:
        logger.error(f"Error saving results: {e}")


# Load the model
print("\n🚀 Starting LULC extraction pipeline...")
model, tokenizer = load_llama_model(CONFIG["model_id"], CONFIG["use_quantization"])

# Load input data
print("\n Loading input data...")
input_data = load_preprocessed_data(CONFIG["input_file"])


✅ Batch processing function defined

🚀 Starting LULC extraction pipeline...
 Loading model: meta-llama/Meta-Llama-3-8B-Instruct
📝 Tokenizer loaded. Vocab size: 128000
🔧 Using 4-bit quantization


INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

ERROR:__main__:Error loading data: [Errno 2] No such file or directory: 'output_entities2.json'


✅ Model loaded successfully on cuda:0

 Loading input data...


In [10]:

# Test with single sentence
if input_data:
    print("\n Testing with single sentence...")
    test_sentence = input_data[2]['sentence']
    print(f"Test sentence: {test_sentence[:150]}...")
    
    # Generate response
    test_response = generate_lulc_extraction(test_sentence, model, tokenizer)
    print(f"\n Raw model response:")
    print(f"'{test_response}'")
    
    # Parse response
    test_parsed = clean_and_parse_response(test_response, test_sentence)
    
    print(f"\n Parsed Results:")
    print(f"   - Entities found: {len(test_parsed['entities'])}")
    for entity in test_parsed['entities']:
        print(f"     • {entity['text']} | {entity['type']}")
    
    print(f"   - Relations found: {len(test_parsed['relations'])}")
    for relation in test_parsed['relations']:
        print(f"     • {relation}")
else:
    print(" No data available for testing")

 No data available for testing


In [20]:
if input_data:
    print(f"\n Processing {min(len(input_data), CONFIG['max_sentences'])} sentences...")
    results = process_sentences_batch(
        input_data, 
        model, 
        tokenizer, 
        max_sentences=CONFIG["max_sentences"]
    )
    
    # Save final results
    final_output_path = f"{CONFIG['output_dir']}/llama3_lulc_extraction_results COT 3 shot  .json"
    save_results(results, final_output_path)
    
    # Print summary statistics
    print("\n Final Statistics:")
    print(f"  - Total sentences processed: {len(results)}")
    
    sentences_with_entities = sum(1 for r in results if r['extracted_entities'])
    sentences_with_relations = sum(1 for r in results if r['extracted_relations'])
    total_entities = sum(len(r['extracted_entities']) for r in results)
    total_relations = sum(len(r['extracted_relations']) for r in results)
    
    print(f"  - Sentences with entities: {sentences_with_entities} ({sentences_with_entities/len(results)*100:.1f}%)")
    print(f"  - Sentences with relations: {sentences_with_relations} ({sentences_with_relations/len(results)*100:.1f}%)")
    print(f"  - Total entities extracted: {total_entities}")
    print(f"  - Total relations extracted: {total_relations}")
    print(f"  - Average entities per sentence: {total_entities/len(results):.2f}")
    print(f"  - Average relations per sentence: {total_relations/len(results):.2f}")

# Cell 13: Analyze results
def analyze_extraction_quality(results: List[Dict]):
    """Analyze the quality of extractions"""
    entity_types = {}
    relation_types = {}
    
    for result in results:
        for entity in result['extracted_entities']:
            entity_type = entity['type']
            entity_types[entity_type] = entity_types.get(entity_type, 0) + 1
        
        for relation in result['extracted_relations']:
            # Extract relation type
            if "--" in relation:
                parts = relation.split("--")
                if len(parts) >= 2:
                    rel_type = parts[1].split("--")[0].strip()
                    relation_types[rel_type] = relation_types.get(rel_type, 0) + 1
    
    print("\n Entity Type Distribution:")
    for entity_type, count in sorted(entity_types.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {entity_type}: {count}")
    
    print("\n Relation Type Distribution:")
    for rel_type, count in sorted(relation_types.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {rel_type}: {count}")

if 'results' in locals():
    analyze_extraction_quality(results)

print("\n LULC extraction pipeline completed successfully!")


 Processing 20 sentences...
 Processing 20 sentences...
📝 Processing 1/20: #text: After the droughts in the 1970s and 1980s, the observ...
    Found 8 entities, 7 relations
📝 Processing 2/20: #text: The forests of West and Central Africa probably origi...
    Found 4 entities, 0 relations
📝 Processing 3/20: )., head: Land use change in the study site, p: ref: 2a, 2b,...
    Found 11 entities, 9 relations
📝 Processing 4/20: Agriculture in this region has been dominated over the past ...
    Found 4 entities, 1 relations
📝 Processing 5/20: Forest land was decreased nearly by half in 2002 compared to...
    Found 4 entities, 4 relations
📝 Processing 6/20: About 26 and 31% of the total area of natural vegetation (fo...
    Found 8 entities, 10 relations
📝 Processing 7/20: Assuming the dynamics recorded in the second period , the am...
    Found 8 entities, 5 relations
📝 Processing 8/20: Conversely the decrease in the herbaceous standing crop, due...
    Found 7 entities, 9 relations
📝 Pro

In [21]:
import csv
import re
from typing import List, Dict, Tuple, Optional

# Define approved relations (from your prompt)
APPROVED_RELATIONS = {
    'CHANGE_TO', 'INCREASES_BY', 'DECREASES_BY', 'CAUSES', 
    'LOCATED_IN', 'OCCURS_DURING', 'MEASURES', 'AFFECTS', 
    'FROM_TO', 'ENABLES', 'CHANGES_TO','DECREASED_BY'
}

def parse_relation(relation_str: str) -> Optional[Tuple[str, str, str, str, str, str]]:
    """
    Parse a relation string to extract source, source_type, relationship, target, target_type, and confidence
    
    Expected format: source:SOURCE_TYPE --RELATIONSHIP-- target:TARGET_TYPE | CONF: confidence
    """
    try:
        # Initialize confidence as empty
        confidence = ""
        
        # Extract confidence if present
        if "| CONF:" in relation_str:
            parts = relation_str.split("| CONF:")
            relation_str = parts[0].strip()
            confidence = parts[1].strip()
        elif "|" in relation_str and relation_str.endswith(("HIGH", "MEDIUM", "LOW")):
            # Handle format: ... | HIGH
            parts = relation_str.rsplit("|", 1)
            relation_str = parts[0].strip()
            confidence = parts[1].strip()
        
        # Split by the relationship marker
        if "--" not in relation_str:
            return None
            
        # Find the relationship type (between --)
        parts = relation_str.split("--")
        if len(parts) < 3:
            return None
            
        # Extract components
        source_part = parts[0].strip()
        relationship = parts[1].strip()
        target_part = parts[2].strip()
        
        # Parse source (format: text:TYPE)
        if ":" not in source_part:
            return None
        source_split = source_part.rsplit(":", 1)
        source = source_split[0].strip()
        source_type = source_split[1].strip()
        
        # Parse target (format: text:TYPE)
        if ":" not in target_part:
            return None
        target_split = target_part.rsplit(":", 1)
        target = target_split[0].strip()
        target_type = target_split[1].strip()
        
        return source, source_type, relationship, target, target_type, confidence
        
    except Exception as e:
        print(f"Error parsing relation: {relation_str} - {e}")
        return None

def is_approved_relation(relationship: str) -> bool:
    """
    Check if a relationship type is in the approved list
    """
    return relationship.upper().strip() in APPROVED_RELATIONS

def save_results_to_csv(results: List[Dict], output_path: str, filter_relations: bool = True):
    """
    Save extraction results to CSV with proper relation parsing and filtering
    """
    
    # Prepare CSV data
    csv_rows = []
    sentence_id = 1
    
    # Statistics tracking
    total_relations = 0
    filtered_relations = 0
    approved_relations = 0
    filtered_out_relations = []  # Track what was filtered
    
    print(f"\n📊 Processing results for CSV export...")
    print(f"🔍 Relation filtering: {'ENABLED' if filter_relations else 'DISABLED'}")
    
    for result in results:
        sentence = result['sentence']
        relations = result.get('extracted_relations', [])
        
        if not relations:
            # Add row even if no relations found
            csv_rows.append({
                'sentenceID': sentence_id,
                'sentence': sentence,
                'source': '',
                'source_type': '',
                'relationship': '',
                'target': '',
                'target_type': '',
                'confidence': ''
            })
        else:
            # Process each relation
            valid_relations = 0
            sentence_filtered = 0
            
            for relation in relations:
                total_relations += 1
                parsed = parse_relation(relation)
                
                if parsed:
                    source, source_type, relationship, target, target_type, confidence = parsed
                    
                    # Apply filtering if enabled
                    if filter_relations and not is_approved_relation(relationship):
                        filtered_relations += 1
                        sentence_filtered += 1
                        filtered_out_relations.append({
                            'sentence_id': sentence_id,
                            'relation': relationship,
                            'full_relation': relation
                        })
                        print(f"  FILTERED: {relationship} (not in approved list)")
                        continue
                    
                    # Add approved relation
                    csv_rows.append({
                        'sentenceID': sentence_id,
                        'sentence': sentence,
                        'source': source,
                        'source_type': source_type,
                        'relationship': relationship,
                        'target': target,
                        'target_type': target_type,
                        'confidence': confidence
                    })
                    valid_relations += 1
                    approved_relations += 1
            
            if valid_relations > 0 or sentence_filtered > 0:
                status = f"✅ {valid_relations} approved"
                if sentence_filtered > 0:
                    status += f",  {sentence_filtered} filtered"
                print(f"  Sentence {sentence_id}: {status}")
        
        sentence_id += 1
    
    # Write to CSV
    if csv_rows:
        with open(output_path, 'w', newline='', encoding='utf-8') as csvfile:
            fieldnames = ['sentenceID', 'sentence', 'source', 'source_type', 'relationship', 
                         'target', 'target_type', 'confidence']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            
            writer.writeheader()
            writer.writerows(csv_rows)
        
        print(f"\n✅ CSV file saved to: {output_path}")
        print(f"   Total rows: {len(csv_rows)}")
        
        # Print comprehensive statistics
        sentences_with_relations = len(set(row['sentenceID'] for row in csv_rows if row['source']))
        
        print(f"\n📈 Processing Statistics:")
        print(f"   - Total sentences: {sentence_id - 1}")
        print(f"   - Total relations found: {total_relations}")
        print(f"   - Approved relations kept: {approved_relations}")
        print(f"   - Relations filtered out: {filtered_relations}")
        print(f"   - Sentences with valid relations: {sentences_with_relations}")
        
        # Show filtering summary
        if filter_relations and filtered_out_relations:
            print(f"\n Filtered Relations Summary:")
            filtered_types = {}
            for item in filtered_out_relations:
                rel_type = item['relation']
                if rel_type not in filtered_types:
                    filtered_types[rel_type] = 0
                filtered_types[rel_type] += 1
            
            for rel_type, count in sorted(filtered_types.items()):
                print(f"   - {rel_type}: {count} times")
        
        # Show approved relations list
        print(f"\n✅ Approved Relations (only these are kept):")
        for rel in sorted(APPROVED_RELATIONS):
            print(f"   - {rel}")
        
        # Show sample of the CSV content
        print(f"\n📋 Sample CSV content (first 5 approved relations):")
        sample_count = 0
        for row in csv_rows:
            if row['source'] and sample_count < 5:
                print(f"   {row['sentenceID']} | {row['source']}:{row['source_type']} "
                      f"--{row['relationship']}-- {row['target']}:{row['target_type']} "
                      f"| CONF: {row['confidence']}")
                sample_count += 1
    else:
        print("⚠️ No data to save to CSV")

# Execute the CSV export with filtering
if 'results' in locals() and results:
    csv_output_path = f"{CONFIG['output_dir']}/llama3_lulc_relations_3shot.csv"
    save_results_to_csv(results, csv_output_path, filter_relations=True)  # Set to False to disable filtering


📊 Processing results for CSV export...
🔍 Relation filtering: ENABLED
  FILTERED: LOST (not in approved list)
  FILTERED: CONSIDERED_AS (not in approved list)
  Sentence 1: ✅ 5 approved,  2 filtered
  Sentence 3: ✅ 9 approved
  FILTERED: DOMINATED_BY (not in approved list)
  Sentence 4: ✅ 0 approved,  1 filtered
  Sentence 5: ✅ 2 approved
  FILTERED: PART_OF (not in approved list)
  Sentence 6: ✅ 8 approved,  1 filtered
  Sentence 7: ✅ 5 approved
  FILTERED: EXTENDS (not in approved list)
  FILTERED: LIMITS (not in approved list)
  FILTERED: LIMITS (not in approved list)
  Sentence 8: ✅ 6 approved,  3 filtered
  FILTERED: AFFECTED_BY (not in approved list)
  FILTERED: APPLIES_TO (not in approved list)
  FILTERED: SCARCE (not in approved list)
  FILTERED: SCARCE (not in approved list)
  FILTERED: SCARCE (not in approved list)
  Sentence 9: ✅ 0 approved,  5 filtered
  Sentence 10: ✅ 6 approved
  Sentence 11: ✅ 1 approved
  FILTERED: DOMINANT (not in approved list)
  Sentence 12: ✅ 2 appr